In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import glob
import os

In [10]:
folder = "/Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands"
print(os.getcwd())
print("Files in folder:", os.listdir(folder))

/Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY
Files in folder: ['tile30S090E.csv', 'tile35S045E.png', 'tile85S005E.png', 'tile70S035E.png', 'tile40S060E.png', 'tile85S140E.png', 'tile70S170E.png', 'tile50S035E.cs', 'tile40S125E.png', 'tile45S170E.cs', 'tile55S170E.cs', 'tile40S035E.cs', 'tile35S100E.png', 'tile80S325E.cs', 'tile55S320E.png', 'tile30S285E.cs', 'tile90S345E.png', 'tile85S260E.cs', 'tile55S265E.png', 'tile65S230E.png', 'tile30S050E.cs', 'tile50S355E.png', 'tile60S300E.png', 'tile35S115E.cs', 'tile50S210E.png', 'tile60S245E.png', 'tile65S290E.csv', 'tile70S095E.csv', 'tile75S335E.cs', 'tile80S070E.png', 'tile45S015E.png', 'tile60S270E.cs', 'tile70S270E.cs', 'tile65S335E.cs', 'tile30S030E.png', 'tile30S175E.png', 'tile40S185E.csv', 'tile75S105E.png', 'tile45S150E.png', 'tile40S310E.csv', 'tile70S345E.csv', 'tile35S075E.cs', 'tile80S090E.cs', 'tile30S130E.cs', 'tile35S335E.csv', 'tile35S270E.csv', 'tile45S280E.png', 'tile40S255E.csv', 'tile70S200E.csv', 'tile65S255E.cs',

In [16]:
folder = "/Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands"
csv_files = glob.glob(os.path.join(folder, "*.csv"))

dfs = []

for f in csv_files:
    try:
        # read CSV, skip malformed lines
        df = pd.read_csv(f, engine='python', skipinitialspace=True, on_bad_lines='skip')
        
        # extract coordinates from the filename
        # e.g., tile30S090E.csv -> Lat=-30, Lon=90
        basename = os.path.basename(f)
        match = re.search(r"tile(\d+)([NS])(\d+)([EW])", basename)
        if match:
            lat = int(match.group(1)) * (-1 if match.group(2) == 'S' else 1)
            lon = int(match.group(3)) * (-1 if match.group(4) == 'W' else 1)
        else:
            lat, lon = None, None
        
        df['Lat'] = lat
        df['Lon'] = lon
        
        dfs.append(df)
    except Exception as e:
        print(f"Failed to read {f}: {e}")

# combine all CSVs
combined_df = pd.concat(dfs, ignore_index=True)

# remove rows without coordinates
combined_df = combined_df.dropna(subset=['Lat', 'Lon'])

# make a GeoDataFrame
gdf = gpd.GeoDataFrame(
    combined_df,
    geometry=gpd.points_from_xy(combined_df['Lon'], combined_df['Lat']),
    crs="EPSG:4326"
)

print(gdf.head())
print(f"Total craters loaded: {len(gdf)}")


Failed to read /Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands/tile30S090E.csv: name 're' is not defined
Failed to read /Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands/tile65S290E.csv: name 're' is not defined
Failed to read /Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands/tile70S095E.csv: name 're' is not defined
Failed to read /Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands/tile40S185E.csv: name 're' is not defined
Failed to read /Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands/tile40S310E.csv: name 're' is not defined
Failed to read /Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands/tile70S345E.csv: name 're' is not defined
Failed to read /Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands/tile35S335E.csv: name 're' is not defined
Failed to read /Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_high

ValueError: No objects to concatenate

In [19]:
import pandas as pd
import geopandas as gpd
import glob, os, re

folder = "/Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/test_southern_highlands"
csv_files = glob.glob(os.path.join(folder, "*.csv"))

data_rows = []

for f in csv_files:
    basename = os.path.basename(f)
    # extract coordinates from filename
    match = re.search(r"tile(\d+)([NS])(\d+)([EW])", basename)
    if match:
        lat = int(match.group(1)) * (-1 if match.group(2) == 'S' else 1)
        lon = int(match.group(3)) * (-1 if match.group(4) == 'W' else 1)
    else:
        lat, lon = None, None
    
    with open(f, 'r', encoding='utf-8', errors='ignore') as file:
        for line in file:
            parts = line.strip().split(",")
            # keep only numeric rows (first value should be a number, not 'Name' etc.)
            if not parts or not parts[0].replace('.', '', 1).isdigit():
                continue
            # add coords
            parts.append(lat)
            parts.append(lon)
            data_rows.append(parts)

# define likely craterstat numeric columns (adjust to what you need)
columns = ["Diameter", "Count", "N_event", "Age", "N(1)", "Lat", "Lon"]

df = pd.DataFrame(data_rows, columns=columns[:len(data_rows[0])])

# convert numbers
for col in ["Diameter","Count","N_event","Age","N(1)","Lat","Lon"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# build GeoDataFrame
gdf = gpd.GeoDataFrame(
    df.dropna(subset=["Lat","Lon"]),
    geometry=gpd.points_from_xy(df["Lon"], df["Lat"]),
    crs="EPSG:4326"
)

print(gdf.head(10))
print(f"✅ Loaded {len(gdf)} crater rows")


ValueError: 7 columns passed, passed data had 28 columns

In [18]:
gdf.head()

,Name,Area,Binning,d_min,d_max,Method,Resurf,N,N_event,Age,...,Source,Extra1,Range,Age2,N2,Age3,N3,Lat,Lon,geometry
0,Name,Area,Binning,d_min,d_max,Method,Resurf,N,N_event,Age,...,Source,,Range,Age,N(1),Age,N(1),-30,90,POINT (90 -30)
1,Name,Area,Binning,d_min,d_max,Method,Resurf,N,N_event,Age,...,Source,,Range,Age,N(1),Age,N(1),-65,290,POINT (290 -65)
2,Name,Area,Binning,d_min,d_max,Method,Resurf,N,N_event,Age,...,Source,,Range,Age,N(1),Age,N(1),-70,95,POINT (95 -70)
3,Name,Area,Binning,d_min,d_max,Method,Resurf,N,N_event,Age,...,Source,,Range,Age,N(1),Age,N(1),-40,185,POINT (185 -40)
4,Name,Area,Binning,d_min,d_max,Method,Resurf,N,N_event,Age,...,Source,,Range,Age,N(1),Age,N(1),-40,310,POINT (310 -40)
